# Private Property Transactions: Modelling Setup
Initial data cleaning, feature engineering, and dataset splitting to support hypothesis testing and future price prediction models.

In [1]:
# Imports and notebook configuration
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

DATA_DIR = Path("../data")
data_path = DATA_DIR / "private_combined.csv"

In [2]:
# Load the combined private transactions dataset
raw_df = pd.read_csv(data_path)
print(f"Loaded {raw_df.shape[0]:,} rows and {raw_df.shape[1]} columns from {data_path.name}.")
raw_df.head()

Loaded 139,317 rows and 17 columns from private_combined.csv.


,Project Name,Transacted Price ($),Area (SQFT),Unit Price ($ PSF),Sale Date,Street Name,Type of Sale,Type of Area,Area (SQM),Unit Price ($ PSM),Nett Price($),Property Type,Number of Units,Tenure,Postal District,Market Segment,Floor Level
0,HARBOUR RESIDENCES,"6,340,000","2,185.09","2,901",Sep-25,PASIR PANJANG ROAD,New Sale,Land,203,"31,232",-,Terrace House,1,Freehold,5,Rest of Central Region,-
1,LANDED HOUSING DEVELOPMENT,"15,200,000","7,812.51","1,946",Sep-25,OCEAN DRIVE,Resale,Land,725.8,"20,942",-,Detached House,1,99 yrs lease commencing from 2004,4,Core Central Region,-
2,BLAIR PLAIN CONSERVATION AREA,"5,900,000","2,037.63","2,896",Sep-25,BLAIR ROAD,Resale,Land,189.3,"31,167",-,Terrace House,1,Freehold,2,Rest of Central Region,-
3,HARBOUR RESIDENCES,"5,630,000","1,983.81","2,838",Sep-25,PASIR PANJANG ROAD,New Sale,Land,184.3,"30,548",-,Terrace House,1,Freehold,5,Rest of Central Region,-
4,HARBOUR RESIDENCES,"5,928,000","2,046.24","2,897",Sep-25,PASIR PANJANG ROAD,New Sale,Land,190.1,"31,184",-,Terrace House,1,Freehold,5,Rest of Central Region,-


In [3]:
# Clean numeric columns and convert to numeric dtypes
clean_df = raw_df.copy()

numeric_columns = [
    "Transacted Price ($)",
    "Area (SQFT)",
    "Unit Price ($ PSF)",
    "Area (SQM)",
    "Unit Price ($ PSM)",
    "Nett Price($)",
    "Number of Units",
]

def parse_numeric(series: pd.Series) -> pd.Series:
    cleaned = series.astype(str).str.strip()
    cleaned = cleaned.str.replace(r"[,$()]", "", regex=True)
    cleaned = cleaned.replace({r"^\s*-+$": np.nan, r"(?i)^nan$": np.nan, "": np.nan}, regex=True)
    return pd.to_numeric(cleaned, errors="coerce")

for col in numeric_columns:
    if col in clean_df.columns:
        clean_df[col] = parse_numeric(clean_df[col])

clean_df[numeric_columns].describe().T

,count,mean,std,min,25%,50%,75%,max
Transacted Price ($),139317.0,2.139910e+06,4.620993e+06,320000.00,1228888.00,1623800.00,2320000.00,8.900000e+08
Area (SQFT),139317.0,1.258353e+03,2.421592e+03,258.34,731.95,1011.82,1323.97,6.198342e+05
Unit Price ($ PSF),139317.0,1.767399e+03,6.106595e+02,120.00,1304.00,1680.00,2167.00,6.593000e+03
Area (SQM),139317.0,1.169038e+02,2.249714e+02,24.00,68.00,94.00,123.00,5.758400e+04
Unit Price ($ PSM),139317.0,1.902429e+04,6.573142e+03,1288.00,14035.00,18089.00,23327.00,7.096400e+04
Nett Price($),246.0,2.304599e+06,2.210020e+06,802286.00,1272032.00,1799300.00,2330300.00,1.446300e+07
Number of Units,139317.0,1.010207e+00,1.394298e+00,1.00,1.00,1.00,1.00,4.460000e+02


In [4]:
# Convert selected columns to categorical dtypes for modelling convenience
categorical_columns = [
    "Property Type",
    "Type of Sale",
    "Type of Area",
    "Postal District",
    "Market Segment",
]

for col in categorical_columns:
    if col in clean_df.columns:
        clean_df[col] = clean_df[col].astype("string").str.strip()
        if col == "Postal District":
            clean_df[col] = clean_df[col].str.zfill(2)
        clean_df[col] = clean_df[col].astype("category")

clean_df[categorical_columns].dtypes

Property Type      category
Type of Sale       category
Type of Area       category
Postal District    category
Market Segment     category
dtype: object

In [5]:
# Parse sale dates and derive time-based helper columns
clean_df["sale_date"] = pd.to_datetime(clean_df["Sale Date"], format="%b-%y", errors="coerce")
clean_df["sale_year"] = clean_df["sale_date"].dt.year
clean_df["sale_month"] = clean_df["sale_date"].dt.month
clean_df["sale_quarter"] = clean_df["sale_date"].dt.to_period("Q")

missing_dates = clean_df["sale_date"].isna().sum()
print(f"Rows with unparsed sale dates: {missing_dates}")
clean_df[["Sale Date", "sale_date", "sale_year", "sale_quarter"]].head()

Rows with unparsed sale dates: 0


,Sale Date,sale_date,sale_year,sale_quarter
0,Sep-25,2025-09-01,2025,2025Q3
1,Sep-25,2025-09-01,2025,2025Q3
2,Sep-25,2025-09-01,2025,2025Q3
3,Sep-25,2025-09-01,2025,2025Q3
4,Sep-25,2025-09-01,2025,2025Q3


In [ ]:
# Build or load an RPI index and create RPI-adjusted price column
reference_year = clean_df["sale_year"].dropna().max()
if pd.isna(reference_year):
    raise ValueError("No valid sale_year values available to anchor RPI adjustment.")

rpi_file = DATA_DIR / "residential_price_index.csv"
if rpi_file.exists():
    rpi_df = pd.read_csv(rpi_file)
    lower_cols = {col.lower(): col for col in rpi_df.columns}
    if "year" not in lower_cols or "rpi" not in lower_cols:
        raise ValueError("RPI reference file must contain 'year' and 'rpi' columns.")
    rpi_df = rpi_df.rename(columns={lower_cols["year"]: "year", lower_cols["rpi"]: "rpi"})
    if any(col.lower() == "period" for col in rpi_df.columns):
        rpi_df = rpi_df.groupby("year", as_index=False)["rpi"].mean()
    rpi_df["year"] = rpi_df["year"].astype(int)
    year_to_rpi = rpi_df.set_index("year")["rpi"]
else:
    warnings.warn(
        "RPI reference file not found; deriving a proxy index from the median unit price ($ PSF) per year.",
        stacklevel=1,
    )
    derived_rpi = (
        clean_df.dropna(subset=["sale_year", "Unit Price ($ PSF)"])
        .groupby("sale_year")["Unit Price ($ PSF)"]
        .median()
        .rename("rpi")
    )
    year_to_rpi = (derived_rpi / derived_rpi.loc[reference_year] * 100)

if reference_year not in year_to_rpi.index:
    raise KeyError(
        f"Reference year {reference_year} is not available in the RPI mapping. Update the RPI data to include this year."
    )

clean_df["rpi_index"] = clean_df["sale_year"].map(year_to_rpi)
base_rpi = year_to_rpi.loc[reference_year]
clean_df["transacted_price_rpi_adjusted"] = np.where(
    clean_df["rpi_index"].notna(),
    clean_df["Transacted Price ($)"] * (base_rpi / clean_df["rpi_index"]),
    np.nan,
)
clean_df[["Transacted Price ($)", "rpi_index", "transacted_price_rpi_adjusted", "sale_year"]].head()

In [ ]:
# Build the modelling-ready dataframe by removing rows missing critical values
required_columns = ["Transacted Price ($)", "sale_year", "transacted_price_rpi_adjusted"]
model_df = clean_df.dropna(subset=required_columns).copy()
print(f"Modelling dataset shape: {model_df.shape}")
model_df[required_columns + ["Property Type", "Type of Sale", "Type of Area"]].head()

In [ ]:
# Train/test split with 80/20 proportion stratified by sale year
target_column = "Transacted Price ($)"
rpi_target = "transacted_price_rpi_adjusted"

feature_columns = [
    col for col in model_df.columns if col not in {target_column, rpi_target}
]

train_df, test_df = train_test_split(
    model_df,
    test_size=0.2,
    stratify=model_df["sale_year"],
    random_state=42,
)

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"Train size: {train_df.shape[0]:,} rows")
print(f"Test size: {test_df.shape[0]:,} rows")
print("Year distribution (train):")
display(train_df["sale_year"].value_counts(normalize=True).sort_index())
print("\nYear distribution (test):")
display(test_df["sale_year"].value_counts(normalize=True).sort_index())

In [ ]:
# Peek at the prepared train and test sets
print("Train sample")
display(train_df.head())
print("Test sample")
display(test_df.head())